Monitoramento e Fairness

In [ ]:
import pandas as pd
import joblib
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

Carregar modelo e vetor já treinados

In [ ]:
modelo_path = "../models/model.joblib"
vetor_path = "../models/vectorizer.joblib"
dados_path = "../data/processed/tweets_limpo.csv"

if not (os.path.exists(modelo_path) and os.path.exists(vetor_path)):
    raise FileNotFoundError("Treine e salve o modelo antes de executar")

model = joblib.load(modelo_path)
vectorizer = joblib.load(vetor_path)
df = pd.read_csv(dados_path)
display(df.head())



Monitoramento

In [ ]:
novos_textos = [
    'Muito ruim, não gostei do atendimento.',
    'A entrega foi sensacional.',
    'Não funcionou, me decepcionei.',
    'Recomendo para todos, nota 10! Top',
    'Recomendo para todos Top',
]

novos_df = pd.DataFrame({f'text': novos_textos})
display(novos_df)



Vetorizar e predição

In [ ]:
novos_vetores = vectorizer.transform(novos_df["text"])
novos_preds = model.predict(novos_vetores)
novos_df['sentimento_predito'] = novos_preds
print(novos_df)

Porcentagem de cada classe

In [ ]:
class_dist = novos_df['sentimento_predito'].value_counts(normalize=True)
print('Distribuição dos sentimentos preditos nos novos dados:')
print(class_dist)

Fairness - Vieses

In [ ]:
df['text_len'] = df['text'].apply(len)
df['len_category'] = pd.cut(df['text_len'], bins=[0,50,150,1000], labels=['curto','medio','longo'])

display(df['len_category'])

Predições conjunto de validação

In [ ]:
vetores = vectorizer.transform(df['text'])
df['pred'] = model.predict(vetores)
display(df['pred'])

Avaliação de acurácia

In [ ]:
for cat in df['len_category'].unique():
    subset = df[df['len_category']== cat]
    if not subset.empty:
        acuracia = (subset['label'] == subset['pred']).mean()
        print(f'Acuracia para os textos {cat}: {acuracia:.2f} N={len(subset)}')

Desafio fairness

In [ ]:
for cat in df['len_category'].unique():
    subset = df[df['len_category'] == cat]
    if not subset.empty:
        print(f'Matriz de confusão para textos {cat}:')
        print(confusion_matrix(subset['label'], subset["pred"]))
        sns.heatmap(confusion_matrix(subset['label'], subset["pred"]),
                    annot=True, fmt='d', cmap="Blues",
                    xticklabels=['Previsto negativo','Previsto Neutro','Previsto positivo'],
                    yticklabels=['Real negativo','Real neutro','Real positivo'])
        plt.title(f'Textos {cat}')
        plt.xlabel('Predito')
        plt.ylabel('Real')
        plt.show()
